In [25]:
import os
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.preprocessing import FunctionTransformer, LabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

In [26]:
# with open(model_path, 'rb') as model_file:
#     best_model = pickle.load(model_file)
# print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
# predictions = best_model.predict_proba(test)[:, 1]
# submission = pd.DataFrame({'id': test_id, 'loan_paid_back': predictions})
# submission_path = os.path.join(MODEL_DIR, f'submission_{model_filename}.csv')
# submission.to_csv(f'submission{model_filename}.csv', index=False)BASE_DIR = Path.cwd().parent
BASE_DIR = Path.cwd().parent
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_FILE = os.path.join(DATA_DIR, "train_raw.csv")
TEST_FILE = os.path.join(DATA_DIR, "test_raw.csv")
MODEL_DIR = os.path.join(BASE_DIR, "models")
df = pd.read_csv(TRAIN_FILE, index_col=0)
df.head()

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
id,,,,,,,,,,,,
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [27]:
train = pd.read_csv(TRAIN_FILE, index_col=0)
test = pd.read_csv(TEST_FILE)
test_id = test['id']
test.drop(columns=['id'])

# Target
y_train = train['loan_paid_back']

In [28]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer
from itertools import combinations, product
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import lightgbm as lgb

# 1. Digits extractor (pierwsza cyfra, suma cyfr, ostatnia cyfra)
def extract_all_digits(df):
    """Extract digits from numeric features"""
    digit_features = {}
    numeric_cols = ['credit_score', 'loan_amount', 'interest_rate', 'annual_income', 'debt_to_income_ratio']

    for col in numeric_cols:
        col_data = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)

        # First digit
        col_int = col_data.astype(int)
        first_digit = np.where(col_int > 0,
            col_int // 10**np.floor(np.log10(np.maximum(col_int, 1))).astype(int), 0)
        digit_features[f'{col}_first_digit'] = first_digit.astype(int)

        # Digit sum
        digit_features[f'{col}_digit_sum'] = col_data.astype(str).apply(
            lambda x: sum(int(d) for d in str(x).replace('.', '') if d.isdigit())
        )

        # Last digit
        digit_features[f'{col}_last_digit'] = (col_data % 10).astype(int)

    return pd.DataFrame(digit_features)

# 2. Digit interactions (pairs + triples)
def digit_interactions(df):
    """Create interactions between digit features"""
    digits = extract_all_digits(df)
    interactions = []

    # Pairs
    for pair in combinations(digits.columns, 2):
        inter = digits[list(pair)].astype(str).agg('_'.join, axis=1)
        interactions.append(inter.rename(f'digit_{pair[0]}_{pair[1]}'))

    # Triples (limited)
    for triple in list(combinations(digits.columns[:6], 3))[:10]:  # Top 10 triples
        inter = digits[list(triple)].astype(str).agg('_'.join, axis=1)
        interactions.append(inter.rename(f'digit_triple_{triple[0][:3]}_{triple[1][:3]}_{triple[2][:3]}'))

    return pd.concat(interactions, axis=1)

# 3. Round features (S5E11 style)
def round_features(df):
    """Round numeric features"""
    rounds = {}
    numeric_cols = ['loan_amount', 'annual_income', 'credit_score']

    for col in numeric_cols:
        rounds[f'{col}_round100'] = np.round(df[col] / 100) * 100
        rounds[f'{col}_floor100'] = np.floor(df[col] / 100) * 100

    return pd.DataFrame(rounds)

# 4. TE/CE of base features
def base_features_te(df, y):
    """Target encode base features"""
    base_cols = ['grade_subgrade', 'employment_status', 'loan_purpose']
    te = TargetEncoder(smooth=10.0)
    return pd.DataFrame(te.fit_transform(df[base_cols], y), columns=[f'{c}_te' for c in base_cols])

# 5. TE using employment_status as target (leak exploit!)
def employment_te(df):
    """TE base features using employment_status as target"""
    employment_map = {'Unemployed':0, 'Employed':1, 'Retired':2}
    emp_target = df['employment_status'].map(employment_map)
    base_cols = ['grade_subgrade', 'loan_purpose', 'education_level']

    te = TargetEncoder(smooth=5.0)
    return pd.DataFrame(te.fit_transform(df[base_cols], emp_target), columns=[f'{c}_emp_te' for c in base_cols])

# 6. Quantile bins + categorical credit_score
def misc_features(df):
    """Quantile bins and categorical versions"""
    misc = {}

    # Quantile bins
    kb = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')
    misc['credit_score_bin'] = kb.fit_transform(df[['credit_score']]).ravel()
    misc['debt_bin'] = kb.fit_transform(df[['debt_to_income_ratio']]).ravel()

    # Categorical credit_score (first 2 digits)
    misc['credit_score_cat'] = (df['credit_score'] // 10).astype(int)

    return pd.DataFrame(misc)

# 7. COMBINE ALL
def ultimate_feature_engineering(df, y=None):
    """All features in one function"""
    features = {}

    # Base interactions
    features.update({'debt_x_credit': df['debt_to_income_ratio'] * df['credit_score']})
    features.update({'debt_log': np.log1p(df['debt_to_income_ratio'].clip(lower=0))})

    # Digits
    digits = extract_all_digits(df)
    features.update(digits.iloc[:, :10].to_dict('series'))  # Top 10 digit features

    # Digit interactions
    digit_int = digit_interactions(df).iloc[:, :15]  # Top 15
    features.update(digit_int.to_dict('series'))

    # Rounds
    rounds = round_features(df)
    features.update(rounds.to_dict('series'))

    # Misc
    misc = misc_features(df)
    features.update(misc.to_dict('series'))

    if y is not None:
        # TE base features
        base_te = base_features_te(df, y)
        features.update(base_te.to_dict('series'))

        # Employment TE
        emp_te = employment_te(df)
        features.update(emp_te.to_dict('series'))

    return pd.DataFrame(features).fillna(0)

# 8. PIPELINE VERSION
feat_eng_transformer = FunctionTransformer(ultimate_feature_engineering)

preprocessor = Pipeline([
    ('feat_eng', feat_eng_transformer),
    ('scale', 'passthrough')  # No scaling for trees
])

In [29]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_val_score
#
# N_JOBS = 2
#
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("classifier", LogisticRegression(
#         max_iter=1000,
#         solver="liblinear",
#         C=1.0,
#     ))
# ])
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#
# scores = cross_val_score(
#     pipe,
#     train,
#     y_train,
#     cv=cv,
#     scoring="roc_auc",
#     n_jobs=N_JOBS
# )
# print("Baseline ROC-AUC:", scores.mean(), "+/-", scores.std())


In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

N_JOBS = 1

# POPRAWIONY preprocessor (z poprzedniego kodu)
pipe = Pipeline([
    ("preprocessor", preprocessor),  # Twój działający preprocessor
    ("classifier", RandomForestClassifier())  # Placeholder
])

# POPRAWIONY search_space - DODANE num_leaves i poprawne parametry
search_space = [
    {
        "classifier": [LGBMClassifier(
            random_state=42,
            device="gpu",
            verbose=-1,  # Wyłącz verbose w GridSearch
            n_jobs=N_JOBS
        )],
        "classifier__n_estimators": [100, 300],
        "classifier__max_depth": [5, 10],
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__num_leaves": [64, 128, 256],  # ← DODANE: LGBM wymaga tego!
        "classifier__subsample": [0.8, 0.9]        # ← DODANE: stabilność
    },
    {
        "classifier": [XGBClassifier(
            random_state=42,
            n_jobs=N_JOBS,
            tree_method='hist',
            verbosity=0,
            predictor='gpu_predictor'

        )],
        "classifier__n_estimators": [100, 300],
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__max_depth": [6, 10]
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gridsearch = GridSearchCV(
    pipe,
    search_space,
    cv=cv,
    scoring='roc_auc',
    n_jobs=1,        # ← KLUCZOWE dla GPU
    verbose=2        # Mniej spamu
)
best_model = gridsearch.fit(train, y_train)


Fitting 5 folds for each of 56 candidates, totalling 280 fits


In [ ]:
import datetime

best_model_name = str(best_model.best_estimator_['classifier'].__class__.__name__)
best_model_score = f"{best_model.best_score_}"
model_filename = f'{best_model_name}{best_model_score}.pkl'
model_folder = os.path.join(MODEL_DIR, str(datetime.datetime.now()))
os.mkdir(model_folder)
model_path = os.path.join(model_folder, model_filename)
with open(model_path, "wb") as model_file:
    pickle.dump(best_model, model_file)

print(f"Najlepszy model: {best_model_name}")
print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
print(f"Ranking CV: {best_model.cv_results_['mean_test_score'][-5:]}")

In [ ]:
with open(model_path, 'rb') as model_file:
    best_model = pickle.load(model_file)
print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
predictions = best_model.predict_proba(test)[:, 1]
submission = pd.DataFrame({'id': test_id, 'loan_paid_back': predictions})
submission_path = os.path.join(model_folder, f'submission_{model_filename}.csv')
submission.to_csv(submission_path, index=False)